# CellPose-SAM — Custom Model Training

Fine-tunes a CellPose-SAM model on your own annotated data.

## Workflow

1. **Export annotations from QuPath** using the scripts in `export_annotations/cellpose_training/`.  
   Each exported image produces a pair of files: `<name>_img.tif` and `<name>_mask.tif`.

2. **Organise your data** — place the exported pairs in the `data/` folder next to this notebook, either flat or in subdirectories (one per dataset):
   ```
   data/
   ├── frame001_img.tif        ← flat: all images in data/
   └── frame001_mask.tif
   ```
   ```
   data/
   ├── experiment_A/           ← multi-dataset: one subfolder per dataset
   │   ├── frame001_img.tif
   │   └── frame001_mask.tif
   └── experiment_B/
       ├── frame001_img.tif
       └── frame001_mask.tif
   ```

3. **Create the train/test split** — run the **Split data** cell below (adjust `test_fraction` if needed), or from a terminal with the `cellpose-sam` environment active:
   ```
   python split_data.py data/
   ```
   Either creates `data/splits/train/` and `data/splits/test/` (90 / 10 % by default).

4. **Edit the configuration cell** — set your model name and adjust hyperparameters if needed.

5. **Run all cells** — the trained model is saved to `models/<model_name>`.

---

> **3D data:** multi-page TIFF stacks are supported without any extra configuration — cellpose auto-detects dimensionality from the arrays. When running inference on 3D stacks, use `model.eval(..., do_3D=True)`.

In [ ]:
from cellpose import io, models, train

## Split data

Run this cell to split your annotated pairs into train and test sets. Adjust `test_fraction` if needed — the defaults (90 / 10 %) are a reasonable starting point.

Skip this cell if you have already split the data using `split_data.py` from the terminal.

In [ ]:
# ── Split configuration ──────────────────────────────────────────────────────
test_fraction  = 0.1   # fraction held out for testing
eval_fraction  = 0.0   # set > 0 to create an additional eval set (0 = disabled)
seed           = 42
overwrite      = False  # set True to replace an existing splits/ directory
# ────────────────────────────────────────────────────────────────────────────

import os, random, shutil, sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
from split_data import detect_layout, link_or_copy

data_dir   = Path("data").resolve()
splits_dir = data_dir / "splits"

if splits_dir.exists():
    if overwrite:
        shutil.rmtree(splits_dir)
        print("Removed existing splits/")
    else:
        raise FileExistsError(
            f"'{splits_dir}' already exists. Set overwrite = True to replace it."
        )

print(f"Scanning '{data_dir}' ...")
try:
    pairs = detect_layout(data_dir)
except SystemExit as e:
    raise RuntimeError(e) from None
print(f"Found {len(pairs)} valid pairs")

if len(pairs) < 2:
    raise ValueError("Need at least 2 pairs to create a split")

random.seed(seed)
random.shuffle(pairs)

n_total     = len(pairs)
n_eval      = round(n_total * eval_fraction) if eval_fraction > 0 else 0
n_remaining = n_total - n_eval
n_test      = max(1, round(n_remaining * test_fraction))
n_train     = n_remaining - n_test

if n_train < 1:
    raise ValueError("Not enough pairs for the requested split fractions")

eval_pairs  = pairs[:n_eval]
test_pairs  = pairs[n_eval : n_eval + n_test]
train_pairs = pairs[n_eval + n_test:]

split_sets = [("train", train_pairs), ("test", test_pairs)]
if n_eval > 0:
    split_sets.append(("eval", eval_pairs))

for split_name, split_pairs in split_sets:
    split_dir = splits_dir / split_name
    split_dir.mkdir(parents=True)
    for img, mask in split_pairs:
        link_or_copy(img, split_dir / img.name)
        link_or_copy(mask, split_dir / mask.name)

summary = f"Train: {n_train} | Test: {n_test}"
if n_eval > 0:
    summary += f" | Eval: {n_eval}"
print(f"{summary}  →  {splits_dir}")

In [ ]:
# ── Edit this cell ──────────────────────────────────────────────────────────
model_name    = "my_cpsam_model"   # name for the saved model
train_dir     = "data/splits/train"
test_dir      = "data/splits/test"
n_epochs      = 200
learning_rate = 1e-5
weight_decay  = 0.1
use_gpu       = True               # set to False to use CPU
plot_format   = "pdf"              # loss plot format: "pdf", "png", or "both"
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
io.logger_setup()

output = io.load_train_test_data(
    train_dir, test_dir,
    image_filter="_img", mask_filter="_mask",
)
images, labels, _, test_images, test_labels, _ = output

model = models.CellposeModel(gpu=use_gpu)

model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=images,
    train_labels=labels,
    test_data=test_images,
    test_labels=test_labels,
    weight_decay=weight_decay,
    learning_rate=learning_rate,
    n_epochs=n_epochs,
    model_name=model_name,
)

print(f"Model saved to: {model_path}")

In [ ]:
from pathlib import Path

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_losses, label="train")
    ax.plot(test_losses, label="test")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    fig.tight_layout()

    base_path = Path("models") / f"{model_name}_loss"
    base_path.parent.mkdir(exist_ok=True)
    saved = []
    if plot_format in ("pdf", "both"):
        fig.savefig(base_path.with_suffix(".pdf"))
        saved.append(base_path.with_suffix(".pdf").name)
    if plot_format in ("png", "both"):
        fig.savefig(base_path.with_suffix(".png"), dpi=150)
        saved.append(base_path.with_suffix(".png").name)

    plt.show()
    print(f"Loss plot saved to: {', '.join(saved)}")
except ImportError:
    print("matplotlib not available — install it to plot losses.")
    print(f"train_losses: {train_losses}")
    print(f"test_losses:  {test_losses}")